In [2]:
import pandas as pd
import json
from pathlib import Path
import csv
import numpy as np

In [49]:
book = '/kaggle/input/casml-dataset/Dataset_RAG (1)/book.pdf'    
queries = '/kaggle/input/casml-dataset/Dataset_RAG (1)/queries.json' 
output = Path('submission.csv')

In [50]:
import os

In [51]:
with open(queries, 'r', encoding='utf-8') as f:
    queries = json.load(f)

In [52]:
queries

[{'query_id': '1', 'question': 'What is the scientific method in psychology?'},
 {'query_id': '2', 'question': 'What are the basic parts of a neuron?'},
 {'query_id': '3', 'question': 'What are the stages of sleep?'},
 {'query_id': '4', 'question': 'What is operant conditioning?'},
 {'query_id': '5', 'question': 'What is problem-solving in psychology?'},
 {'query_id': '6', 'question': 'What are the three stages of memory?'},
 {'query_id': '7', 'question': 'What are the key components of emotion?'},
 {'query_id': '8',
  'question': 'What are the major personality traits in the Five Factor Model?'},
 {'query_id': '9', 'question': 'What is social psychology?'},
 {'query_id': '10', 'question': 'What is the sociocultural model in therapy?'},
 {'query_id': '11', 'question': 'What is the history of psychology?'},
 {'query_id': '12', 'question': 'Who were Wilhelm Wundt and William James?'},
 {'query_id': '13', 'question': 'What is functionalism in psychology?'},
 {'query_id': '14',
  'question

In [54]:
!pip install pdfplumber


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [53]:
import pdfplumber

In [55]:
pages = []  
with pdfplumber.open(book) as pdf:
    for i, page in enumerate(pdf.pages, start=1):
        txt = page.extract_text(x_tolerance=2, y_tolerance=2) or ""
        txt = "\n".join([ln.strip() for ln in txt.splitlines() if ln.strip()])
        pages.append({"page_no": i, "text": txt})

In [56]:
import re

In [57]:
section_for_page = {}  # page_no -> section_title (if found)
current_section = None

section_heading_patterns = [
    r'^\s*CHAPTER\s+\d+', r'^\s*Chapter\s+\d+'
]

for p in pages:
    lines = p['text'].splitlines()
    found = None
    for ln in lines[:6]:
        if any(re.search(pat, ln, flags=re.IGNORECASE) for pat in section_heading_patterns):
            found = ln.strip()
            break
    # fallback: если нет "Chapter", то если первая строка короткая и выглядит как заголовок (есть буквы, не предложение)
    if not found and lines:
        first = lines[0].strip()
        if 3 <= len(first) <= 120 and (first.isupper() or (len(first.split())<=6 and first[0].isupper())):
            found = first
    if found:
        current_section = found
    section_for_page[p['page_no']] = current_section or "Unknown"


In [12]:
section_for_page

{1: 'Unknown',
 2: 'Unknown',
 3: 'Psychology 2e',
 4: 'OpenStax',
 5: 'OPENSTAX',
 6: 'Study where you want, what',
 7: 'CHAPTER 1',
 8: 'CHAPTER 1',
 9: 'CHAPTER 8',
 10: 'CHAPTER 8',
 11: 'CHAPTER 15',
 12: 'Access for free at openstax.org',
 13: 'Preface 1',
 14: 'Preface 1',
 15: 'Preface 3',
 16: 'Preface 3',
 17: 'Preface 5',
 18: 'Preface 5',
 19: 'Preface 5',
 20: 'Preface 5',
 21: 'Preface 5',
 22: 'Preface 5',
 23: 'Preface 5',
 24: 'Preface 5',
 25: 'Preface 5',
 26: 'Preface 5',
 27: 'Preface 5',
 28: 'Preface 5',
 29: 'Preface 5',
 30: 'Preface 5',
 31: 'Preface 5',
 32: 'Preface 5',
 33: 'Preface 5',
 34: 'Preface 5',
 35: 'Preface 5',
 36: 'Preface 5',
 37: 'Preface 5',
 38: 'Preface 5',
 39: 'Preface 5',
 40: 'Preface 5',
 41: 'Preface 5',
 42: 'Preface 5',
 43: 'Preface 5',
 44: 'Preface 5',
 45: 'Preface 5',
 46: 'Preface 5',
 47: 'Preface 5',
 48: 'Preface 5',
 49: 'Preface 5',
 50: 'Preface 5',
 51: 'Preface 5',
 52: 'Preface 5',
 53: 'Preface 5',
 54: 'Preface 5',

In [58]:
from sentence_transformers import SentenceTransformer

In [59]:
embed_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(embed_model_name)

In [60]:
!pip install -q semantic-text-splitter nltk

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 54.9 MB/s eta 0:00:0000:0100:01


In [61]:
import nltk
from semantic_text_splitter import TextSplitter

In [62]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [63]:

# --------------------------------------------------
# 🔹 Функция для семантического разбиения текста
# --------------------------------------------------
def semantic_split_text(text, max_tokens=200):
    """
    Делит текст по смыслу (предложения, абзацы) вместо фиксированной длины.
    Использует semantic-text-splitter + nltk fallback.
    """
    try:
        splitter = TextSplitter.from_tiktoken_encoder(
            model_name="gpt-3.5-turbo",  # совместимо с Qwen и аналогами
            chunk_size=max_tokens,
            chunk_overlap=50
        )
        chunks = splitter.chunks(text)
    except Exception:
        # fallback — просто делим по предложениям
        sentences = nltk.sent_tokenize(text)
        chunks, current = [], ""
        for s in sentences:
            if len(current) + len(s) > max_tokens * 4:  # примерно
                chunks.append(current.strip())
                current = s
            else:
                current += " " + s
        if current.strip():
            chunks.append(current.strip())
    return chunks

# --------------------------------------------------
# 🔹 Применяем к страницам / документам
# --------------------------------------------------
semantic_chunks = []
for p in tqdm(pages, desc="Semantic chunking"):
    page_no = p['page_no']
    text = p['text']
    sections = semantic_split_text(text, max_tokens=180)

    for chunk in sections:
        semantic_chunks.append({
            "page_no": page_no,
            "text": chunk,
            "section": section_for_page.get(page_no, "Unknown")
        })

print(f"✅ Всего чанков: {len(semantic_chunks)}")


Semantic chunking:   0%|          | 0/753 [00:00<?, ?it/s]

✅ Всего чанков: 3701


In [39]:
def chunk_page_text(page_text, page_no, section, max_chars=1200, overlap=200):
    chunks = []
    txt = page_text
    if not txt:
        return []
    i = 0
    L = len(txt)
    while i < L:
        part = txt[i: i + max_chars]
        chunks.append({
            "page_no": page_no,
            "section": section,
            "text": part
        })
        i += max_chars - overlap
    return chunks

chunks = []
for p in pages:
    chunks.extend(chunk_page_text(p['text'], p['page_no'], section_for_page[p['page_no']], max_chars=1200, overlap=200))

In [40]:
len(chunks)

2642

In [17]:
import tqdm

In [66]:
semantic_chunks[10]

{'page_no': 7,
 'text': '36\n2.2 Approaches to Research 41\n2.3 Analyzing Findings 48\n2.4 Ethics 59\nKey Terms 63\nSummary 64\nReview Questions 66\nCritical Thinking Questions 69\nPersonal Application Questions 70\nCHAPTER 3\nBiopsychology\n71\nIntroduction 71\n3.1 Human Genetics 72\n3.2 Cells of the Nervous System 78\n3.3 Parts of the Nervous System 84\n3.4 The Brain and Spinal Cord 86\n3.5 The Endocrine System 97\nKey Terms 100\nSummary 102\nReview Questions 103\nCritical Thinking Questions 106\nPersonal Application Questions 106\nCHAPTER 4\nStates of Consciousness\n109\nIntroduction 109\n4.1 What Is Consciousness? 110',
 'section': 'CHAPTER 1'}

In [67]:
texts = [c['text'] for c in semantic_chunks]
batch = 128
emb_list = []
for i in tqdm(range(0, len(texts), batch), desc="Embedding"):
    sub = texts[i:i+batch]
    em = embedder.encode(sub, show_progress_bar=False, normalize_embeddings=True)
    emb_list.append(em)
embeddings = np.vstack(emb_list).astype('float32')  # shape = (N, dim)
print(embeddings.shape)

Embedding:   0%|          | 0/29 [00:00<?, ?it/s]

(3701, 384)


In [68]:
embeddings[0]

array([ 5.80102652e-02,  6.18371516e-02,  3.87067394e-03,  9.48799625e-02,
       -5.69939949e-02,  9.56913980e-04,  4.33431976e-02,  1.12138363e-02,
        1.02925554e-01, -3.24501172e-02,  1.30841334e-04,  1.64516363e-03,
        6.59960974e-03,  1.65128410e-02,  4.31401618e-02, -1.51521293e-02,
        5.27998284e-02,  3.99326533e-03, -1.09881893e-01, -1.25417039e-02,
       -8.35515186e-03, -3.02352048e-02,  4.78644520e-02, -2.77162511e-02,
       -1.00386038e-01,  1.33871529e-02, -2.81945691e-02, -1.25867516e-01,
        1.19232181e-02, -4.79622278e-03, -4.69937688e-03, -5.83323068e-04,
        3.63250934e-02,  5.62652154e-03, -4.91327718e-02, -1.18629718e-02,
       -1.91735290e-02,  1.13749363e-01, -4.96258819e-03,  3.33097909e-04,
       -6.32093102e-02, -8.91806092e-03,  5.92967262e-03, -3.34820487e-02,
       -1.25231994e-02, -7.33031556e-02, -2.82205213e-02, -3.48970219e-02,
       -1.06925957e-01, -5.22217415e-02, -8.34633932e-02, -1.30079463e-02,
       -9.68259014e-03,  

In [20]:
!pip install faiss-cpu

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [21]:
import faiss

In [69]:
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)   
index.add(embeddings)

In [79]:
index

<faiss.swigfaiss_avx512.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x7c514d089a40> >

In [70]:
embeddings.shape

(3701, 384)

In [71]:
print(index.ntotal)

3701


In [80]:
meta = [{"page_no": c["page_no"], "section": c["section"], "text": c["text"]} for c in chunks]

In [28]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [29]:
gen_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

In [73]:
tokenizer = AutoTokenizer.from_pretrained(gen_model_name)
model = AutoModelForCausalLM.from_pretrained(
    gen_model_name,
    torch_dtype="auto")

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0,  
    do_sample=False,
    max_new_tokens=400
)

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [81]:
def build_prompt(question, contexts):
    ctx_text = ""
    for i,c in enumerate(contexts, start=1):
        ctx_text += f"[CONTEXT {i}] SECTION: {c['section']} | PAGE: {c['page_no']}\n{c['text']}\n\n"
    prompt = (
        "You are an expert psychology assistant. "
        "Answer ONLY based on the provided textbook context. "
        "If the information is missing, say 'No data found in the textbook.'\n\n"
        f"Question: {question}\n\n"
        f"Context:\n{ctx_text}\n\n"
        "Give a concise answer in English. "
        "Then output JSON with fields 'sections' and 'pages' for the cited sources.\n"
        "Format example: {\"sections\": [\"...\"], \"pages\": [\"...\"]}\n"
    )
    return prompt

In [92]:
def retrieve(query, top_k=5):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype('float32')
    D, I = index.search(q_emb, top_k)
    idxs = I[0].tolist()
    results = [meta[i] for i in idxs]
    return results, q_emb, D, I, idxs

In [ ]:
'What is the scientific method in psychology?'

In [93]:
results, q_emb, D, I, idx = retrieve('What is the scientific method in psychology?', top_k=5)

In [95]:
q_emb

array([[ 3.31251360e-02,  7.72634894e-02, -6.05272129e-02,
         5.58931977e-02, -4.29528691e-02,  3.40702012e-02,
        -4.68931273e-02,  8.37683529e-02,  1.73148196e-02,
         5.77760600e-02, -4.08849586e-03, -3.55258060e-04,
        -3.69375013e-02,  4.95445356e-02, -5.59994057e-02,
        -1.49594778e-02, -4.08309661e-02,  1.36495447e-02,
         4.11540568e-02, -4.58180085e-02,  1.56285353e-02,
        -4.89531159e-02,  4.34310250e-02, -1.45325204e-02,
        -3.63519415e-02,  6.56595966e-03, -3.96012291e-02,
        -5.97417057e-02,  6.11775927e-03,  1.51607729e-02,
         8.40146020e-02,  6.84476718e-02,  1.99613273e-02,
         1.33484667e-02, -1.23243809e-01,  6.92226961e-02,
        -5.47430553e-02,  7.71259964e-02,  6.03782199e-02,
         7.61651397e-02, -7.55049214e-02, -1.05087794e-02,
         5.04769199e-03, -3.72153637e-03,  2.40645241e-02,
        -4.30079289e-02, -3.55206169e-02, -1.39321201e-02,
        -2.73424909e-02, -1.63208935e-02, -1.06247388e-0

In [94]:
results

[{'page_no': 24,
  'section': 'Preface 5',
  'text': 'h a sensory experience can be broken down into individual parts, how those parts relate to each other\nas a whole is often what the individual responds to in perception. For example, a song may be made up of\nindividual notes played by different instruments, but the real nature of the song is perceived in the\ncombinations of these notes as they form the melody, rhythm, and harmony. In many ways, this particular\nperspective would have directly contradicted Wundt’s ideas of structuralism (Thorne & Henley, 2005).\nUnfortunately, in moving to the United States, these scientists were forced to abandon much of their work and\nwere unable to continue to conduct research on a large scale. These factors along with the rise of behaviorism\n(described next) in the United States prevented principles of Gestalt psychology from being as influential in\nthe United States as they had been in their native Germany (Thorne & Henley, 2005). Despite t

In [33]:
!pip install -q sentence-transformers accelerate

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [34]:
from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader
import numpy as np
import random
from tqdm.auto import tqdm

In [76]:
# Можно дообучить (если есть пары)
model_name = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
reranker = CrossEncoder(model_name, num_labels=1)
# или: reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank_candidates(query, candidates, reranker, batch_size=32):
    pairs = [(query, c['text']) for c in candidates]
    scores = reranker.predict(pairs, batch_size=batch_size, show_progress_bar=False)
    for c, s in zip(candidates, scores):
        c['score_rerank'] = float(s)
    return sorted(candidates, key=lambda x: x['score_rerank'], reverse=True)


In [85]:
import torch, gc, os, time
from tqdm.auto import tqdm

# 💡 включаем настройку, которая помогает избежать фрагментации памяти
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

results_out = []

# 🧠 ключевые параметры
TOP_K_RETRIEVE = 30        # меньше кандидатов на rerank
TOP_AFTER_RERANK = 5       # меньше контекста → меньше токенов
MAX_NEW_TOKENS = 300       # сократим длину генерации
RERANK_BATCH = 8           # размер батча для CrossEncoder
CLEAR_EVERY = 2            # чистить память каждые N запросов

for q_idx, q in enumerate(tqdm(queries, desc="Questions"), start=1):
    try:
        qid = q.get('query_id') or q.get('ID') or q.get('id')
        question = q['question']

        # 🔹 Шаг 1: первичный retrieve по FAISS
        retrieved = retrieve(question, top_k=TOP_K_RETRIEVE)

        # 🔹 Шаг 2: rerank (ограниченный batch size)
        reranked = rerank_candidates(question, retrieved, reranker, batch_size=RERANK_BATCH)
        top_retrieved = reranked[:TOP_AFTER_RERANK]

        # 🔹 Шаг 3: формируем контекст
        context_concat = "\n\n".join([r['text'] for r in top_retrieved])
        prompt = build_prompt(question, top_retrieved)

        # 🔹 Шаг 4: генерация ответа
        with torch.inference_mode():  # выключает градиенты → экономит VRAM
            gen_out = generator(
                prompt,
                max_new_tokens=MAX_NEW_TOKENS,
                num_return_sequences=1,
                do_sample=False
            )
        gen = gen_out[0]['generated_text']

        # 🔹 Шаг 5: извлекаем JSON (если есть)
        json_refs = None
        try:
            last_brace = gen.rfind('{')
            if last_brace != -1:
                possible = gen[last_brace:]
                json_refs = json.loads(possible)
        except Exception:
            json_refs = None

        if not json_refs:
            sections = list({r['section'] for r in top_retrieved if r['section']})
            pages = list({str(r['page_no']) for r in top_retrieved})
            json_refs = {"sections": sections, "pages": pages}

        answer_text = gen[:gen.rfind('{')].strip() if '{' in gen else gen.strip()

        results_out.append({
            "ID": qid,
            "context": context_concat.replace('"', "'"),
            "answer": answer_text.replace('"', "'"),
            "references": json.dumps(json_refs, ensure_ascii=False)
        })

    except torch.cuda.OutOfMemoryError:
        print(f"\n⚠️ OOM on question {q_idx}: {question[:60]}...")
        torch.cuda.empty_cache()
        gc.collect()
        time.sleep(2)
        continue

    # 🔹 Шаг 6: периодическая очистка кэша
    if q_idx % CLEAR_EVERY == 0:
        torch.cuda.empty_cache()
        gc.collect()
        time.sleep(0.5)

print(f"\n✅ Done. Processed {len(results_out)} questions.")

Questions:   0%|          | 0/50 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p'


✅ Done. Processed 50 questions.


In [86]:
df = pd.DataFrame(results_out)[['ID','context','answer','references']]
df.to_csv('submission.csv', index=False, quoting=csv.QUOTE_MINIMAL)

In [87]:
df

,ID,context,answer,references
0,1,ing\nempirical method method for acquiring kno...,You are an expert psychology assistant. Answer...,"{""sections"": [""Preface 5""], ""pages"": [""42"", ""4..."
1,2,128 4 • States of Consciousness\n(Figure 4.15)...,You are an expert psychology assistant. Answer...,"{""sections"": [""Preface 5""], ""pages"": [""138"", ""..."
2,3,"164 5 • Sensation and Perception\nloss, hearin...",You are an expert psychology assistant. Answer...,"{""sections"": [""SECTION: Preface 5 | PAGE: 176""..."
3,4,8.2 • Parts of the Brain Involved with Memory ...,You are an expert psychology assistant. Answer...,"{""sections"": [""Preface 5""], ""pages"": [""264"", ""..."
4,5,ith the\ncaregiver.\nAnother potential problem...,You are an expert psychology assistant. Answer...,"{""sections"": [""Preface 5""], ""pages"": [""57"", ""3..."
5,6,10 • Key Terms 353\nKey Terms\nanorexia nervos...,You are an expert psychology assistant. Answer...,"{""sections"": [""Preface 5""], ""pages"": [""397"", ""..."
6,7,"11.7 • Trait Theorists 383\nneuroticism, which...",You are an expert psychology assistant. Answer...,"{""sections"": [""Preface 5"", ""Industrial-Organiz..."
7,8,"sical activity, and sleep quality) (Boehm & Ku...",You are an expert psychology assistant. Answer...,"{""sections"": [""CHAPTER 15"", ""Industrial-Organi..."
8,9,38 2 • Psychological Research\napplied behavio...,You are an expert psychology assistant. Answer...,"{""sections"": [""Preface 5"", ""Industrial-Organiz..."
9,10,38 2 • Psychological Research\napplied behavio...,You are an expert psychology assistant. Answer...,"{""sections"": [""Preface 5""], ""pages"": [""34"", ""5..."
